# Chapter 5: Pretraining

Code from Sebastian Raschka's *Build a Large Language Model (From Scratch)*. Each cell has the exercise, the book code it needs, and my code.

In [ ]:
# Setup
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
BOOK = ROOT / "resources/books/llm"
MODELS_DIR = str(ROOT / "work/llm/gpt2")
BOOK_CODE = BOOK / "ch05/01_main-chapter-code"
sys.path.insert(0, str(BOOK_CODE))

import os
import tiktoken
import torch
from previous_chapters import GPTModel, create_dataloader_v1, generate_text_simple
from gpt_train import train_model_simple, calc_loss_loader, text_to_token_ids, token_ids_to_text
from gpt_generate import generate, load_weights_into_gpt
from gpt_download import download_and_load_gpt2

# Book code: ch05 gpt_train.py and section 5.5
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256,  # Shortened context length (orig: 1024)
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-key-value bias
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")

with open(BOOK / "ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

# Trains the chapter's model once (10 epochs), then reuses the saved file. Delete it to retrain.
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

if not os.path.exists("model_and_optimizer.pth"):
    torch.manual_seed(123)
    train_losses, val_losses, tokens_seen = train_model_simple(
        model, train_loader, val_loader, optimizer, device,
        num_epochs=10, eval_freq=5, eval_iter=5,
        start_context="Every effort moves you", tokenizer=tokenizer
    )
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, "model_and_optimizer.pth")

checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval();


In [ ]:
# Exercise 5.1: Temperature-scaled softmax scores and sampling probabilities
# Use print_sampled_tokens to print the sampling frequencies of softmax probabilities
# scaled with the temperatures below. How often is "pizza" sampled in each case? Can you
# think of a faster and more accurate way to find how often it is sampled?

# Book code: ch05 section 5.3.1
import torch

vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")


def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


temperatures = [1, 0.1, 5]  # Original, higher, and lower temperature
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

# Alex's code starts here


In [ ]:
# Exercise 5.2: Different temperature and top-k settings
# Experiment with different temperature and top-k settings. From what you observe, which
# applications would want lower settings? Which would want higher ones?

# Book code: ch05 section 5.3.3
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# Alex's code starts here


In [ ]:
# Exercise 5.3: Deterministic behavior in the decoding functions
# What combinations of settings for the generate function force deterministic behavior,
# disabling random sampling so it always produces the same output, like
# generate_text_simple?

# Book code: ch04 gpt.py and ch05 section 5.3.3
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

# Alex's code starts here


In [ ]:
# Exercise 5.4: Continued pretraining
# After saving the weights, load the model and optimizer in a new session or notebook
# and continue pretraining for one more epoch with train_model_simple.

# Book code: ch05 section 5.4
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

# Alex's code starts here


In [ ]:
# Exercise 5.5: Training and validation set losses of the pretrained model
# Calculate the training and validation set losses of GPTModel with OpenAI's pretrained
# weights on "The Verdict" dataset.

# Book code: ch05 section 5.5
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

settings, params = download_and_load_gpt2(model_size="124M", models_dir=MODELS_DIR)

gpt = GPTModel(NEW_CONFIG)
gpt.eval()
load_weights_into_gpt(gpt, params)
gpt.to(device);

# Alex's code starts here


In [ ]:
# Exercise 5.6: Trying larger models
# Experiment with other GPT-2 sizes, such as the largest 1,558M model, and compare the
# generated text with the 124M model. The 1,558M download is about 6 GB.

# Book code: ch05 section 5.5
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

settings, params = download_and_load_gpt2(model_size="124M", models_dir=MODELS_DIR)

gpt = GPTModel(NEW_CONFIG)
gpt.eval()
load_weights_into_gpt(gpt, params)
gpt.to(device);

# Alex's code starts here
